In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they exist in current working directory per task)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to get compact timestamp strings in UTC+8
def ts_range(df):
    if 'timestamp' not in df.columns or df['timestamp'].dropna().empty:
        return (None, None)
    mn = int(df['timestamp'].min())
    mx = int(df['timestamp'].max())
    mn_dt = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    mx_dt = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    return (mn_dt, mx_dt)

# Summaries and top-20 unique lists
metric_min_ts, metric_max_ts = ts_range(metric_df)
metric_cmdbs = pd.Series(pd.unique(metric_df['cmdb_id'].dropna()))[:20].to_frame(name='cmdb_id')
metric_kpis = pd.Series(pd.unique(metric_df['kpi_name'].dropna()))[:20].to_frame(name='kpi_name')
metric_summary = pd.DataFrame([{
    'file':'metric.csv',
    'rows': len(metric_df),
    'min_timestamp_UTC+8': metric_min_ts,
    'max_timestamp_UTC+8': metric_max_ts,
    'unique_cmdb_count': metric_df['cmdb_id'].nunique(dropna=True),
    'unique_kpi_count': metric_df['kpi_name'].nunique(dropna=True)
}])

trace_min_ts, trace_max_ts = ts_range(trace_df)
trace_cmdbs = pd.Series(pd.unique(trace_df['cmdb_id'].dropna()))[:20].to_frame(name='cmdb_id')
trace_kpis = pd.Series(pd.unique(trace_df['trace_name'].dropna()))[:20].to_frame(name='trace_name')
trace_summary = pd.DataFrame([{
    'file':'trace.csv',
    'rows': len(trace_df),
    'min_timestamp_UTC+8': trace_min_ts,
    'max_timestamp_UTC+8': trace_max_ts,
    'unique_cmdb_count': trace_df['cmdb_id'].nunique(dropna=True),
    'unique_trace_count': trace_df['trace_name'].nunique(dropna=True)
}])

log_min_ts, log_max_ts = ts_range(log_df)
log_cmdbs = pd.Series(pd.unique(log_df['cmdb_id'].dropna()))[:20].to_frame(name='cmdb_id')
log_kpis = pd.Series(pd.unique(log_df['log_name'].dropna()))[:20].to_frame(name='log_name')
log_summary = pd.DataFrame([{
    'file':'log.csv',
    'rows': len(log_df),
    'min_timestamp_UTC+8': log_min_ts,
    'max_timestamp_UTC+8': log_max_ts,
    'unique_cmdb_count': log_df['cmdb_id'].nunique(dropna=True),
    'unique_logname_count': log_df['log_name'].nunique(dropna=True)
}])

error_min_ts, error_max_ts = ts_range(error_df)
error_cmdbs = pd.Series(pd.unique(error_df['cmdb_id'].dropna()))[:20].to_frame(name='cmdb_id')
# For messages, return up to 20 distinct samples, truncated for compactness
if 'message' in error_df.columns:
    unique_msgs = pd.Series(pd.unique(error_df['message'].dropna()))[:20]
    # truncate to 200 chars to keep compact
    unique_msgs = unique_msgs.apply(lambda s: (s[:197] + '...') if len(s)>200 else s).to_frame(name='message_sample')
else:
    unique_msgs = pd.DataFrame(columns=['message_sample'])
error_summary = pd.DataFrame([{
    'file':'error_logs.csv',
    'rows': len(error_df),
    'min_timestamp_UTC+8': error_min_ts,
    'max_timestamp_UTC+8': error_max_ts,
    'unique_cmdb_count': error_df['cmdb_id'].nunique(dropna=True),
    'unique_message_samples': len(unique_msgs)
}])

# Display compact summaries and up-to-20 lists (use variables rather than print)
metric_summary, metric_cmdbs.head(20), metric_kpis.head(20), \
trace_summary, trace_cmdbs.head(20), trace_kpis.head(20), \
log_summary, log_cmdbs.head(20), log_kpis.head(20), \
error_summary, error_cmdbs.head(20), unique_msgs.head(20)

```
Out[1]:
```
metric_cmdb_list = metric_cmdbs['cmdb_id'].astype(str).tolist()[:20]
metric_kpi_list = metric_kpis['kpi_name'].astype(str).tolist()[:20]
trace_cmdb_list = trace_cmdbs['cmdb_id'].astype(str).tolist()[:20]
trace_name_list = trace_kpis['trace_name'].astype(str).tolist()[:20]
log_cmdb_list = log_cmdbs['cmdb_id'].astype(str).tolist()[:20]
log_name_list = log_kpis['log_name'].astype(str).tolist()[:20]
error_msg_list = unique_msgs['message_sample'].astype(str).tolist()[:20] if 'unique_msgs' in globals() else []

summary = (
    "Summary of telemetry files:\n\n"
    "1) metric.csv:\n"
    f"   - Total rows: 27,957\n"
    f"   - Time range (UTC+8): 2021-03-10 15:00:00 to 2021-03-10 15:30:00\n"
    f"   - Unique cmdb_id count: 29 (samples: {', '.join(metric_cmdb_list)})\n"
    f"   - Unique KPI count: 369 (sample KPI names: {', '.join(metric_kpi_list)})\n\n"
    "2) trace.csv:\n"
    f"   - Total rows: 6,861\n"
    f"   - Time range (UTC+8): 2021-03-10 15:00:00 to 2021-03-10 15:29:00\n"
    f"   - Unique cmdb_id count: 12 (samples: {', '.join(trace_cmdb_list)})\n"
    f"   - Unique trace name count: 72 (sample trace names: {', '.join(trace_name_list)})\n\n"
    "3) log.csv:\n"
    f"   - Total rows: 522\n"
    f"   - Time range (UTC+8): 2021-03-10 15:00:00 to 2021-03-10 15:30:00\n"
    f"   - Unique cmdb_id count: 8 (samples: {', '.join(log_cmdb_list)})\n"
    f"   - Unique log name count: 3 (log names: {', '.join(log_name_list)})\n\n"
    "4) error_logs.csv:\n"
    "   - Total rows: 0 (no timestamps, no messages)\n\n"
    "Notes: metric.csv and trace.csv contain extensive KPI/trace types and many hosts; log.csv is smaller. No error log records are present."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows      min_timestamp_UTC+8      max_timestamp_UTC+8  unique_cmdb_count  unique_kpi_count
0  metric.csv  27957  2021-03-10 15:00:00 CST  2021-03-10 15:30:00 CST                 29               369,           cmdb_id
0            IG01
1            IG02
2            MG01
3            MG02
4         Mysql01
5         Mysql02
6         Redis01
7         Redis02
8    ServiceTest1
9   ServiceTest10
10  ServiceTest11
11   ServiceTest2
12   ServiceTest3
13   ServiceTest4
14   ServiceTest5
15   ServiceTest6
16   ServiceTest7
17   ServiceTest8
18   ServiceTest9
19       Tomcat01,                                           kpi_name
0         JVM-Memory_7778_JVM_Memory_HeapMemoryMax
1        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed
2      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed
3                  JVM-Runtime_7778_JVM_JVM_Uptime
4         JVM-Threads_7778_JVM_ThreadCount_Threads
5                  OSLinux-CPU_CPU-0_SingleCpuUtil
6                  OSLinux-CPU_CPU-2_SingleCpuUtil
7                       OSLinux-CPU_CPU_CPUCpuUtil
8                          OSLinux-CPU_CPU_CPULoad
9                       OSLinux-CPU_CPU_CPUSysTime
10                     OSLinux-CPU_CPU_CPUUserTime
11                          OSLinux-CPU_CPU_CPUWio
12                     OSLinux-CPU_CPU_CPUidleutil
13  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
14          OSLinux-OSLinux_MEMORY_MEMORY_CacheMem
15        OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem
16    OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
17           OSLinux-OSLinux_MEMORY_MEMORY_UserMem
18  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
19    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT,         file  rows      min_timestamp_UTC+8      max_timestamp_UTC+8  unique_cmdb_count  unique_trace_count
0  trace.csv  6861  2021-03-10 15:00:00 CST  2021-03-10 15:29:00 CST                 12                  72,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4   Tomcat01
5   Tomcat02
6   Tomcat03
7   Tomcat04
8   dockerA1
9   dockerA2
10  dockerB1
11  dockerB2,                            trace_name
0            trace.self.duration_mean
1             trace.self.duration_p95
2                trace.self.row_count
3     trace.to_Tomcat01.duration_mean
4      trace.to_Tomcat01.duration_p95
5         trace.to_Tomcat01.row_count
6     trace.to_Tomcat02.duration_mean
7      trace.to_Tomcat02.duration_p95
8         trace.to_Tomcat02.row_count
9     trace.to_Tomcat03.duration_mean
10     trace.to_Tomcat03.duration_p95
11        trace.to_Tomcat03.row_count
12    trace.to_Tomcat04.duration_mean
13     trace.to_Tomcat04.duration_p95
14        trace.to_Tomcat04.row_count
15  trace.from_Tomcat01.duration_mean
16   trace.from_Tomcat01.duration_p95
17      trace.from_Tomcat01.row_count
18  trace.from_Tomcat02.duration_mean
19   trace.from_Tomcat02.duration_p95,       file  rows      min_timestamp_UTC+8      max_timestamp_UTC+8  unique_cmdb_count  unique_logname_count
0  log.csv   522  2021-03-10 15:00:00 CST  2021-03-10 15:30:00 CST                  8                     3,     cmdb_id
0   Mysql01
1  Tomcat01
2  Tomcat02
3  Tomcat03
4  Tomcat04
5  apache01
6  apache02
7   Mysql02,            log_name
0   log.error_count
1     log.row_count
2  log.gc_pause_max,              file  rows min_timestamp_UTC+8 max_timestamp_UTC+8  unique_cmdb_count  unique_message_samples
0  error_logs.csv     0                None                None                  0                       0, Empty DataFrame
Columns: [cmdb_id]
Index: [], Empty DataFrame
Columns: [message_sample]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df and tz from previous kernel state if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Candidate components
candidates = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# 2) Filter metric.csv to candidate components (all available timestamps) -> used for global thresholds
metrics_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Ensure 'value' numeric
metrics_candidates['value'] = pd.to_numeric(metrics_candidates['value'], errors='coerce')

# 3) Compute global P95 and P5 per (cmdb_id, kpi_name) using all available rows for those components
thresholds = metrics_candidates.groupby(['cmdb_id','kpi_name'])['value'].agg(
    P5=lambda x: x.quantile(0.05),
    P95=lambda x: x.quantile(0.95)
).reset_index()

# 4) Filter to incident time window March 10, 2021 15:00:00 to 15:30:00 (UTC+8) and keep only candidates
window_start_local = pd.Timestamp('2021-03-10 15:00:00', tz=tz)
window_end_local   = pd.Timestamp('2021-03-10 15:30:00', tz=tz)

# Convert local window to UTC epoch seconds (timestamps in files are treated as UTC seconds)
window_start_epoch = int(window_start_local.tz_convert('UTC').timestamp())
window_end_epoch   = int(window_end_local.tz_convert('UTC').timestamp())

metrics_window = metrics_candidates[
    (metric_df['timestamp'] >= window_start_epoch) & (metric_df['timestamp'] <= window_end_epoch)
].copy()

# Count full_time_window_row_count per (cmdb_id,kpi_name)
window_counts = metrics_window.groupby(['cmdb_id','kpi_name']).size().reset_index(name='full_time_window_row_count')

# 5) Detect anomalies in the window by comparing to precomputed thresholds (P95/P5)
# Merge thresholds into windowed data
metrics_window = metrics_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Drop rows with missing thresholds or values
metrics_window = metrics_window.dropna(subset=['value','P5','P95'])

# Mark anomalies
metrics_window['is_upper'] = metrics_window['value'] >= metrics_window['P95']
metrics_window['is_lower'] = metrics_window['value'] <= metrics_window['P5']

# 6) Aggregate anomalies per (cmdb_id, kpi_name)
# Upper anomalies aggregation
upper = metrics_window[metrics_window['is_upper']].copy()
if not upper.empty:
    upper_agg = upper.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('timestamp','min'),
        max_value = ('value','max'),
        P95 = ('P95','first')
    ).reset_index()
    upper_agg['anomaly_type'] = 'upper'
    upper_agg['threshold_used'] = upper_agg['P95']
    upper_agg['severity'] = upper_agg['max_value'] / upper_agg['P95']
    upper_agg = upper_agg.merge(window_counts, on=['cmdb_id','kpi_name'], how='left').fillna({'full_time_window_row_count':0})
else:
    upper_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','max_value','P95','anomaly_type','threshold_used','severity','full_time_window_row_count'])

# Lower anomalies aggregation
lower = metrics_window[metrics_window['is_lower']].copy()
if not lower.empty:
    lower_agg = lower.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('timestamp','min'),
        min_value = ('value','min'),
        P5 = ('P5','first')
    ).reset_index()
    lower_agg['anomaly_type'] = 'lower'
    lower_agg['threshold_used'] = lower_agg['P5']
    # Avoid division by zero: if min_value == 0, set severity as inf
    lower_agg['severity'] = lower_agg.apply(lambda r: (r['P5'] / r['min_value']) if r['min_value'] != 0 else float('inf'), axis=1)
    lower_agg = lower_agg.merge(window_counts, on=['cmdb_id','kpi_name'], how='left').fillna({'full_time_window_row_count':0})
    # Align column names with upper_agg for concatenation
    lower_agg = lower_agg.rename(columns={'min_value':'min_value', 'P5':'P5'})
else:
    lower_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','min_value','P5','anomaly_type','threshold_used','severity','full_time_window_row_count'])

# Normalize columns for concatenation: ensure both have max_value/min_value column present
if 'max_value' not in lower_agg.columns:
    lower_agg['max_value'] = pd.NA
if 'min_value' not in upper_agg.columns:
    upper_agg['min_value'] = pd.NA

# Select and standardize columns
upper_sel = upper_agg[['cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_ts','max_value','min_value','threshold_used','severity','full_time_window_row_count']]
lower_sel = lower_agg[['cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_ts','max_value','min_value','threshold_used','severity','full_time_window_row_count']]

anomalies_agg = pd.concat([upper_sel, lower_sel], ignore_index=True, sort=False)

# Convert earliest_anomaly_ts to UTC+8 readable string for display but keep epoch for any sorting
if not anomalies_agg.empty:
    anomalies_agg['earliest_anomaly_ts_local'] = anomalies_agg['earliest_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
else:
    anomalies_agg['earliest_anomaly_ts_local'] = pd.Series(dtype=str)

# Calculate severity numeric and sort
anomalies_agg['severity'] = pd.to_numeric(anomalies_agg['severity'], errors='coerce').fillna(0)
anomalies_agg = anomalies_agg.sort_values(by='severity', ascending=False).reset_index(drop=True)

# Prepare final compact columns and limit to top 20
anomalies_compact = anomalies_agg[[
    'cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_ts_local',
    'max_value','min_value','threshold_used','severity','full_time_window_row_count'
]].head(20)

# 7) Per-component summary: distinct KPI anomalies count and earliest anomaly timestamp among its KPIs
if not anomalies_agg.empty:
    comp_summary = anomalies_agg.groupby('cmdb_id').agg(
        distinct_kpi_anomalies = ('kpi_name', lambda x: x.nunique()),
        earliest_anomaly_ts = ('earliest_anomaly_ts', 'min')
    ).reset_index()
    comp_summary['earliest_anomaly_ts_local'] = comp_summary['earliest_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
else:
    comp_summary = pd.DataFrame(columns=['cmdb_id','distinct_kpi_anomalies','earliest_anomaly_ts','earliest_anomaly_ts_local'])

comp_summary = comp_summary.sort_values(by='distinct_kpi_anomalies', ascending=False).head(20)

# Final compact outputs (variables shown, not printed)
anomalies_compact, comp_summary

```
Out[2]:
```
summary = (
    "Anomaly scan summary (metric.csv, candidate components, window 2021-03-10 15:00:00–15:30:00 UTC+8):\n"
    "- Many KPIs show anomalies across the candidate components. Earliest detected anomaly timestamps are at the window start (2021-03-10 15:00:00 UTC+8).\n"
    "- Top per-component distinct KPI anomaly counts (descending): Mysql01:224, Mysql02:224, Redis01:115, Tomcat03:111, Redis02:111, Tomcat02:109, Tomcat04:109, Tomcat01:106, MG01:94, MG02:92, IG01:88, IG02:88, apache01:80, apache02:80.\n"
    "- The compact top anomalies list contains mostly 'lower' anomalies (many KPIs reporting value==0) and a few 'upper' ones. Examples: apache02 shows multiple lower-bound anomalies (many values 0); apache01 has an 'upper' anomaly on a TCP-FIN-WAIT KPI. Several anomalies have threshold values (P5 or P95) equal to 0, producing infinite severity ratios (division-by-zero situation).\n"
    "- Interpretation / data-quality note: numerous thresholds (P5 or P95) are zero for some KPI series, and many observed values are zero in the window. This leads to infinite severity scores and suggests either genuine zero-valued metrics (e.g., error counts, swap I/O) or that the KPI series contains zeros or sparse data; treat infinite severities cautiously and inspect raw series for those KPIs before acting.\n"
    "- Next recommended steps: inspect the raw time series for top components (Mysql01/Mysql02/Redis01 and top Tomcats) and the KPIs with threshold==0 or many zero values to determine whether zeros are valid measurements or instrumentation issues; prioritize KPIs with non-zero thresholds and high severity ratios for operational investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name anomaly_type  anomaly_count earliest_anomaly_ts_local  max_value  min_value  threshold_used  severity  full_time_window_row_count
0   apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...        lower             15   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          15
1   apache02                       OSLinux-OSLinux_SWAP_SWAP_So        lower             15   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          15
2   apache02                       OSLinux-OSLinux_SWAP_SWAP_Si        lower             15   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          15
3   apache01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT        upper             30   2021-03-10 15:00:00 CST        1.0        NaN             0.0       inf                          30
4   Tomcat04            OSLinux-OSLinux_NETWORK_ens160_NETInErr        lower             10   2021-03-10 15:02:00 CST        NaN        0.0             0.0       inf                          10
5   Tomcat04         OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc        lower             10   2021-03-10 15:03:00 CST        NaN        0.0             0.0       inf                          10
6   Tomcat04           OSLinux-OSLinux_NETWORK_ens160_NETOutErr        lower             10   2021-03-10 15:02:00 CST        NaN        0.0             0.0       inf                          10
7   Tomcat04       OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc        lower             10   2021-03-10 15:03:00 CST        NaN        0.0             0.0       inf                          10
8   Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT        lower             20   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          20
9   Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT        lower             20   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          20
10  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps        lower              6   2021-03-10 15:06:00 CST        NaN        0.0             0.0       inf                          10
11  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter        lower             31   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          31
12  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc        lower              9   2021-03-10 15:05:00 CST        NaN        0.0             0.0       inf                           9
13  Tomcat04    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent        lower              9   2021-03-10 15:05:00 CST        NaN        0.0             0.0       inf                           9
14  Tomcat04                       OSLinux-OSLinux_SWAP_SWAP_Si        lower              9   2021-03-10 15:01:00 CST        NaN        0.0             0.0       inf                           9
15  Tomcat04                       OSLinux-OSLinux_SWAP_SWAP_So        lower              9   2021-03-10 15:01:00 CST        NaN        0.0             0.0       inf                           9
16  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...        lower             31   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          31
17  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...        lower             31   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          31
18  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...        lower             31   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          31
19  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...        lower             31   2021-03-10 15:00:00 CST        NaN        0.0             0.0       inf                          31,      cmdb_id  distinct_kpi_anomalies  earliest_anomaly_ts earliest_anomaly_ts_local
4    Mysql01                     224           1615359600   2021-03-10 15:00:00 CST
5    Mysql02                     224           1615359600   2021-03-10 15:00:00 CST
6    Redis01                     115           1615359600   2021-03-10 15:00:00 CST
7    Redis02                     111           1615359600   2021-03-10 15:00:00 CST
10  Tomcat03                     111           1615359600   2021-03-10 15:00:00 CST
9   Tomcat02                     109           1615359600   2021-03-10 15:00:00 CST
11  Tomcat04                     109           1615359600   2021-03-10 15:00:00 CST
8   Tomcat01                     106           1615359600   2021-03-10 15:00:00 CST
2       MG01                      94           1615359600   2021-03-10 15:00:00 CST
3       MG02                      92           1615359600   2021-03-10 15:00:00 CST
0       IG01                      88           1615359600   2021-03-10 15:00:00 CST
1       IG02                      88           1615359600   2021-03-10 15:00:00 CST
12  apache01                      80           1615359600   2021-03-10 15:00:00 CST
13  apache02                      80           1615359600   2021-03-10 15:00:00 CST)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse anomalies_agg and tz from previous kernel state
try:
    anomalies_agg
except NameError:
    raise RuntimeError("Required cached variable 'anomalies_agg' not found in kernel state.")

# Ensure earliest_anomaly_ts_local exists; if not, create it
if 'earliest_anomaly_ts_local' not in anomalies_agg.columns:
    anomalies_agg['earliest_anomaly_ts_local'] = anomalies_agg['earliest_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

# Normalize numeric columns
anomalies_agg['threshold_used'] = pd.to_numeric(anomalies_agg['threshold_used'], errors='coerce')
anomalies_agg['severity'] = pd.to_numeric(anomalies_agg['severity'], errors='coerce')

# 1) Filter: threshold_used > 0, severity finite and >= 1.2
severe = anomalies_agg[
    (anomalies_agg['threshold_used'] > 0) &
    (np.isfinite(anomalies_agg['severity'])) &
    (anomalies_agg['severity'] >= 1.2)
].copy()

# 2) Prepare compact rows with extremal_value
if not severe.empty:
    severe['extremal_value'] = severe.apply(
        lambda r: r['max_value'] if r['anomaly_type']=='upper' else r['min_value'],
        axis=1
    )
else:
    severe['extremal_value'] = pd.Series(dtype=float)

severe_compact = severe[[
    'cmdb_id',
    'kpi_name',
    'anomaly_type',
    'anomaly_count',
    'earliest_anomaly_ts_local',
    'extremal_value',
    'threshold_used',
    'severity',
    'full_time_window_row_count'
]].sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# 3) Per-component summary (top 10 components)
if not severe.empty:
    comp_severe = severe.groupby('cmdb_id').agg(
        count_of_severe_kpi_anomalies = ('kpi_name', 'nunique'),
        earliest_severe_anom_ts = ('earliest_anomaly_ts', 'min')
    ).reset_index()
    comp_severe['earliest_severe_anomaly_timestamp_local'] = comp_severe['earliest_severe_anom_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    comp_severe_top10 = comp_severe.sort_values(by='count_of_severe_kpi_anomalies', ascending=False).head(10)[
        ['cmdb_id','count_of_severe_kpi_anomalies','earliest_severe_anomaly_timestamp_local']
    ].reset_index(drop=True)
else:
    comp_severe_top10 = pd.DataFrame(columns=['cmdb_id','count_of_severe_kpi_anomalies','earliest_severe_anomaly_timestamp_local'])

# Return compact results (variables, not printed)
severe_compact, comp_severe_top10

```
Out[3]:
```
summary = (
    "Refined severe-anomaly summary (threshold>0, finite severity>=1.2) — compact:\n\n"
    f"- Total severe anomaly rows returned: {len(severe_compact)} (showing up to 20).\n\n"
    "Top example anomalies (by severity):\n"
    f"1) {severe_compact.loc[0,'cmdb_id']} | {severe_compact.loc[0,'kpi_name']} | {severe_compact.loc[0,'anomaly_type']} | "
    f"extremal_value={severe_compact.loc[0,'extremal_value']} | threshold={severe_compact.loc[0,'threshold_used']:.6f} | "
    f"severity={severe_compact.loc[0,'severity']:.2f} | earliest={severe_compact.loc[0,'earliest_anomaly_ts_local']}\n"
    f"2) {severe_compact.loc[1,'cmdb_id']} | {severe_compact.loc[1,'kpi_name']} | {severe_compact.loc[1,'anomaly_type']} | "
    f"extremal_value={severe_compact.loc[1,'extremal_value']} | threshold={severe_compact.loc[1,'threshold_used']:.6f} | "
    f"severity={severe_compact.loc[1,'severity']:.2f} | earliest={severe_compact.loc[1,'earliest_anomaly_ts_local']}\n"
    f"3) {severe_compact.loc[2,'cmdb_id']} | {severe_compact.loc[2,'kpi_name']} | {severe_compact.loc[2,'anomaly_type']} | "
    f"extremal_value={severe_compact.loc[2,'extremal_value']} | threshold={severe_compact.loc[2,'threshold_used']:.6f} | "
    f"severity={severe_compact.loc[2,'severity']:.2f} | earliest={severe_compact.loc[2,'earliest_anomaly_ts_local']}\n\n"
    "Pattern highlights and operational notes:\n"
    "- Mysql02 and Mysql01 have the largest counts of severe KPI anomalies (48 and 47 respectively). Many of these are disk-related (DSK*) and InnoDB lock/time metrics.\n"
    "- Tomcat01 shows a large lower-bound CPU-idle anomaly (severity ~35), indicating CPU became heavily utilized (low idle) around 15:16 UTC+8.\n"
    "- Several components (Redis, Apache, Tomcats, MG/IG) also surface multiple severe KPIs but with smaller per-KPI severities.\n\n"
    "Recommended next actions (prioritized):\n"
    "1) Investigate Mysql01/Mysql02: check disk I/O (sda/sdb/sdc DSK*), InnoDB row lock/time metrics and query contention during 15:00–15:30.\n"
    "2) Inspect Tomcat01 CPU usage and processes at ~15:16 (high CPU usage implied by very low CPU-idle).\n"
    "3) Review MG02 and apache01/apache02 CPU WIO and TCP state KPIs (some high severities present).\n"
    "4) For each candidate KPI, open the raw time series around the window to confirm anomalies and rule out instrumentation zeros or short spikes before remediation.\n\n"
    "If you want, I can now (a) show the detailed severe rows (up to 20) or (b) extract the raw time series for any selected (cmdb_id,kpi_name) for deeper inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name anomaly_type  anomaly_count earliest_anomaly_ts_local  extremal_value  threshold_used   severity  full_time_window_row_count
0   Tomcat01                    OSLinux-CPU_CPU-0_SingleCpuidle        lower              1   2021-03-10 15:16:00 CST        1.096500       38.666540  35.263602                           9
1       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT        upper              1   2021-03-10 15:09:00 CST        1.000000        0.050000  20.000000                          20
2    Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time        lower              2   2021-03-10 15:14:00 CST        0.016700        0.152500   9.131737                          21
3    Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps        lower              1   2021-03-10 15:08:00 CST       20.000000      137.600000   6.880000                          15
4       MG02                             OSLinux-CPU_CPU_CPUWio        upper              1   2021-03-10 15:09:00 CST        4.468900        0.840100   5.319486                          19
5   apache01                             OSLinux-CPU_CPU_CPUWio        upper              2   2021-03-10 15:07:00 CST        2.869800        0.547250   5.244038                          31
6       IG01                             OSLinux-CPU_CPU_CPUWio        upper              2   2021-03-10 15:08:00 CST        4.041700        0.846305   4.775704                          31
7    Mysql02             Mysql-MySQL_3306_Innodb Row Lock Waits        lower              2   2021-03-10 15:14:00 CST        0.016700        0.067800   4.059880                          21
8    Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time        upper              2   2021-03-10 15:01:00 CST       68.101700       19.433300   3.504382                          21
9    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead        upper              1   2021-03-10 15:08:00 CST        0.133333        0.040000   3.333333                          15
10   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...        upper              1   2021-03-10 15:13:00 CST        0.400000        0.120000   3.333333                          15
11   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        upper              1   2021-03-10 15:25:00 CST       21.200000        6.360000   3.333333                          15
12   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps        upper              1   2021-03-10 15:25:00 CST        5.000000        1.500000   3.333333                          15
13   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...        upper              1   2021-03-10 15:28:00 CST        0.800000        0.240000   3.333333                          15
14   Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps        upper              1   2021-03-10 15:25:00 CST        4.000000        1.200000   3.333333                          15
15   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps        upper              1   2021-03-10 15:08:00 CST        0.016667        0.005000   3.333333                          15
16  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps        upper              1   2021-03-10 15:10:00 CST        1.000000        0.300000   3.333333                          15
17   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        upper              1   2021-03-10 15:25:00 CST        0.450000        0.135000   3.333333                          15
18   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps        upper              1   2021-03-10 15:25:00 CST        5.000000        1.500000   3.333333                          15
19  apache01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps        upper              1   2021-03-10 15:03:00 CST        1.000000        0.300000   3.333333                          15,     cmdb_id  count_of_severe_kpi_anomalies earliest_severe_anomaly_timestamp_local
0   Mysql02                             48                 2021-03-10 15:00:00 CST
1   Mysql01                             47                 2021-03-10 15:00:00 CST
2  Tomcat01                             15                 2021-03-10 15:00:00 CST
3   Redis01                             13                 2021-03-10 15:00:00 CST
4  apache02                             13                 2021-03-10 15:00:00 CST
5  Tomcat04                             10                 2021-03-10 15:06:00 CST
6  Tomcat03                              9                 2021-03-10 15:08:00 CST
7  Tomcat02                              8                 2021-03-10 15:00:00 CST
8  apache01                              8                 2021-03-10 15:03:00 CST
9      IG02                              8                 2021-03-10 15:00:00 CST)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached trace_df and tz if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Candidate trace components
trace_candidates = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','IG01','IG02','MG01','MG02','apache01','apache02']

# 2) Compute global P95 and P5 per (cmdb_id, trace_name) using full series (no time filter) but limited to candidate components
trace_full_candidates = trace_df[trace_df['cmdb_id'].isin(trace_candidates)].copy()
thresholds_trace = trace_full_candidates.groupby(['cmdb_id','trace_name'])['value'].agg(
    P5=lambda x: x.quantile(0.05),
    P95=lambda x: x.quantile(0.95)
).reset_index()

# 3) Define two UTC+8 windows
windowA_start_local = pd.Timestamp('2021-03-10 15:00:00', tz=tz)
windowA_end_local   = pd.Timestamp('2021-03-10 15:05:00', tz=tz)
windowB_start_local = pd.Timestamp('2021-03-10 15:15:00', tz=tz)
windowB_end_local   = pd.Timestamp('2021-03-10 15:18:00', tz=tz)

# Convert local windows to UTC epoch seconds (trace timestamps are epoch seconds)
windowA_start_epoch = int(windowA_start_local.tz_convert('UTC').timestamp())
windowA_end_epoch   = int(windowA_end_local.tz_convert('UTC').timestamp())
windowB_start_epoch = int(windowB_start_local.tz_convert('UTC').timestamp())
windowB_end_epoch   = int(windowB_end_local.tz_convert('UTC').timestamp())

# Helper to process one window
def analyze_window(window_label, start_epoch, end_epoch):
    # 4) Filter trace rows to candidates and to the window
    window_rows = trace_full_candidates[
        (trace_full_candidates['timestamp'] >= start_epoch) &
        (trace_full_candidates['timestamp'] <= end_epoch)
    ].copy()
    # full_window_row_count per (cmdb_id, trace_name)
    window_counts = window_rows.groupby(['cmdb_id','trace_name']).size().reset_index(name='full_window_row_count')
    # 5) Merge thresholds (computed on full series) into window rows
    w = window_rows.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left')
    w = w.dropna(subset=['value'])  # keep rows with numeric values
    if w.empty:
        return pd.DataFrame(columns=[
            'window_label','cmdb_id','trace_name','anomaly_type','anomaly_count',
            'earliest_anomaly_ts','extremal_value','threshold_used','severity','full_window_row_count'
        ])
    # duration anomalies: trace_name contains 'duration' and value >= P95
    is_duration = w['trace_name'].str.contains('duration', case=False, na=False)
    duration_rows = w[is_duration & (w['value'] >= w['P95'])].copy()
    if not duration_rows.empty:
        dur_agg = duration_rows.groupby(['cmdb_id','trace_name']).agg(
            anomaly_count = ('value','size'),
            earliest_anomaly_ts = ('timestamp','min'),
            extremal_value = ('value','max'),
            threshold_used = ('P95','first')
        ).reset_index()
        # severity = extremal_value / P95
        dur_agg['severity'] = dur_agg['extremal_value'] / dur_agg['threshold_used']
        dur_agg['anomaly_type'] = 'duration'
    else:
        dur_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','extremal_value','threshold_used','severity','anomaly_type'])
    # row_count drop anomalies: trace_name contains 'row_count' and value <= P5
    is_rowcount = w['trace_name'].str.contains('row_count', case=False, na=False)
    rc_rows = w[is_rowcount & (w['value'] <= w['P5'])].copy()
    if not rc_rows.empty:
        rc_agg = rc_rows.groupby(['cmdb_id','trace_name']).agg(
            anomaly_count = ('value','size'),
            earliest_anomaly_ts = ('timestamp','min'),
            extremal_value = ('value','min'),
            threshold_used = ('P5','first')
        ).reset_index()
        # severity = P5 / extremal_value (guard div by zero)
        rc_agg['severity'] = rc_agg.apply(lambda r: (r['threshold_used'] / r['extremal_value']) if r['extremal_value'] not in (0, None, np.nan) else np.inf, axis=1)
        rc_agg['anomaly_type'] = 'row_count_drop'
    else:
        rc_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','extremal_value','threshold_used','severity','anomaly_type'])
    # concat anomalies
    combined = pd.concat([dur_agg, rc_agg], ignore_index=True, sort=False)
    if combined.empty:
        return combined
    # attach full_window_row_count
    combined = combined.merge(window_counts, on=['cmdb_id','trace_name'], how='left').fillna({'full_window_row_count':0})
    # add window label
    combined['window_label'] = window_label
    return combined[[
        'window_label','cmdb_id','trace_name','anomaly_type','anomaly_count',
        'earliest_anomaly_ts','extremal_value','threshold_used','severity','full_window_row_count'
    ]]

# Analyze windows A and B and cache intermediate tables
anomalies_A = analyze_window('A', windowA_start_epoch, windowA_end_epoch)
anomalies_B = analyze_window('B', windowB_start_epoch, windowB_end_epoch)

# 6) Combine anomalies and prepare compact display
anomalies_both = pd.concat([anomalies_A, anomalies_B], ignore_index=True, sort=False)
if not anomalies_both.empty:
    # convert earliest_anomaly_ts to local readable string
    anomalies_both['earliest_anomaly_timestamp_local'] = anomalies_both['earliest_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # sort by severity desc and limit to top 20
    anomalies_compact = anomalies_both.sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)
else:
    anomalies_compact = pd.DataFrame(columns=[
        'window_label','cmdb_id','trace_name','anomaly_type','anomaly_count',
        'earliest_anomaly_timestamp_local','extremal_value','threshold_used','severity','full_window_row_count'
    ])

# 7) Per-component compact summary: by window_label and cmdb_id
if not anomalies_both.empty:
    comp_summary = anomalies_both.groupby(['window_label','cmdb_id']).agg(
        total_anomaly_count = ('anomaly_count','sum'),
        earliest_anomaly_ts = ('earliest_anomaly_ts','min')
    ).reset_index()
    comp_summary['earliest_anomaly_timestamp_local'] = comp_summary['earliest_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    comp_summary = comp_summary.sort_values(['window_label','total_anomaly_count'], ascending=[True, False]).head(20).reset_index(drop=True)
else:
    comp_summary = pd.DataFrame(columns=['window_label','cmdb_id','total_anomaly_count','earliest_anomaly_timestamp_local'])

# Final compact outputs (variables for display)
# anomalies_compact: include displayed columns per requirement
if not anomalies_compact.empty:
    anomalies_display = anomalies_compact[[
        'window_label','cmdb_id','trace_name','anomaly_type','anomaly_count',
        'earliest_anomaly_timestamp_local','extremal_value','threshold_used','severity','full_window_row_count'
    ]]
else:
    anomalies_display = anomalies_compact

anomalies_display, comp_summary.head(20)

```
Out[4]:
```
Summary of trace anomaly scan (trace.csv), candidate components, two UTC+8 windows:

Overview
- Two windows analyzed:
  - Window A: 2021-03-10 15:00–15:05 (UTC+8)
  - Window B: 2021-03-10 15:15–15:18 (UTC+8)
- Global P5/P95 thresholds were computed from the full trace series per (cmdb_id, trace_name).
- Detected anomalies are primarily traffic drops (row_count <= P5). No large duration (latency >= P95) anomalies rose into the top results.

Top anomaly highlights (Window A dominates)
- Highest-severity anomalies (sorted by severity): many row_count_drop events in Window A at ~15:02.
  - MG01 → trace.to_dockerA2.row_count: extremal row_count 82 vs P5=131.70, severity ≈ 1.61 (1 anomalous point, window row count 6).
  - MG01 → trace.to_dockerA1.row_count: 93 vs P5=123.85, severity ≈ 1.33.
  - MG02 → trace.to_dockerA1.row_count: 104 vs P5=127.50, severity ≈ 1.23.
  - Tomcat04 → trace.self.row_count: 2118 vs P5=2589.65, severity ≈ 1.22.
  - Multiple symmetric traces (to/from relationships among MG, IG, Tomcat) show single-point drops at 15:02 with severity ~1.09–1.17.
- Window B shows fewer events and lower counts; MG02 and Tomcat02 appear with some row_count drops around 15:15–15:16.

Per-component compact summary (top entries)
- Window A top components by total anomaly count:
  - Tomcat04: 11 anomalies (earliest 15:00)
  - MG01: 10 anomalies (earliest 15:00)
  - Tomcat02: 10 anomalies (earliest 15:00)
  - MG02: 8 anomalies (earliest ~15:01)
  - Tomcat01, Tomcat03: 8 each (earliest 15:00)
  - IG01/IG02 also show multiple row_count_drop entries
- Window B: MG02 (5), Tomcat02 (4), MG01 (3), Tomcat04 (3) are the most affected in that window.

Interpretation and recommended next steps
- Pattern: a short-lived traffic drop concentrated around ~15:02 in Window A affecting many inter-service traces (Tomcat↔MG/IG/docker). This looks like a brief throughput/traffic reduction rather than sustained high latency.
- Action items (prioritized):
  1. Inspect raw row_count time series for the top traces (e.g., MG01 → dockerA2, Tomcat04 self row_count) around 15:00–15:05 to confirm the drop shape and duration.
  2. Check application logs and ingress/load-balancer metrics at ~15:02 for signs of request throttling, brief network issues, or spikes in error counts.
  3. Correlate with metric anomalies (CPU, disk, network) on MG/Tomcat hosts at the same timestamps to find potential resource causes.
  4. If confirmed as traffic loss (not instrumentation), trace request flows upstream (clients/load-balancers) and downstream (datastores) to find the root cause.
- Window B anomalies are smaller and fewer—treat as lower priority but include them in correlation checks.

If you want, I can now extract the raw trace time series for any of the top (cmdb_id, trace_name) pairs (e.g., MG01 → trace.to_dockerA2.row_count or Tomcat04 → trace.self.row_count) for a detailed inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(   window_label   cmdb_id                     trace_name    anomaly_type  anomaly_count earliest_anomaly_timestamp_local  extremal_value  threshold_used  severity  full_window_row_count
0             A      MG01    trace.to_dockerA2.row_count  row_count_drop              1          2021-03-10 15:02:00 CST            82.0          131.70  1.606098                      6
1             A      MG01    trace.to_dockerA1.row_count  row_count_drop              1          2021-03-10 15:02:00 CST            93.0          123.85  1.331720                      6
2             A      MG02    trace.to_dockerA1.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           104.0          127.50  1.225962                      6
3             A  Tomcat04           trace.self.row_count  row_count_drop              1          2021-03-10 15:02:00 CST          2118.0         2589.65  1.222686                      6
4             A      IG02    trace.to_Tomcat02.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           162.0          191.45  1.181790                      6
5             A  Tomcat02      trace.from_IG02.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           162.0          191.45  1.181790                      6
6             A      MG02  trace.from_Tomcat04.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           167.0          195.80  1.172455                      6
7             A  Tomcat04        trace.to_MG02.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           167.0          195.80  1.172455                      6
8             A  Tomcat04      trace.from_IG01.row_count  row_count_drop              2          2021-03-10 15:02:00 CST           151.0          175.45  1.161921                      6
9             A      IG01    trace.to_Tomcat04.row_count  row_count_drop              2          2021-03-10 15:02:00 CST           151.0          175.45  1.161921                      6
10            A      MG01  trace.from_Tomcat04.row_count  row_count_drop              2          2021-03-10 15:02:00 CST           160.0          183.25  1.145312                      6
11            A  Tomcat04        trace.to_MG01.row_count  row_count_drop              2          2021-03-10 15:02:00 CST           160.0          183.25  1.145312                      6
12            A      IG02           trace.self.row_count  row_count_drop              1          2021-03-10 15:02:00 CST          1400.0         1535.10  1.096500                      6
13            A      MG02  trace.from_Tomcat02.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           170.0          186.15  1.095000                      6
14            A  Tomcat02        trace.to_MG02.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           170.0          186.15  1.095000                      6
15            A  Tomcat02        trace.to_MG01.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           166.0          181.40  1.092771                      6
16            A      MG01  trace.from_Tomcat02.row_count  row_count_drop              1          2021-03-10 15:02:00 CST           166.0          181.40  1.092771                      6
17            A      IG01           trace.self.row_count  row_count_drop              1          2021-03-10 15:02:00 CST          1330.0         1439.30  1.082180                      6
18            A  Tomcat02           trace.self.row_count  row_count_drop              1          2021-03-10 15:02:00 CST          2235.0         2415.90  1.080940                      6
19            A  Tomcat01           trace.self.row_count  row_count_drop              1          2021-03-10 15:02:00 CST          2326.0         2514.10  1.080868                      6,    window_label   cmdb_id  total_anomaly_count  earliest_anomaly_ts earliest_anomaly_timestamp_local
0             A  Tomcat04                   11           1615359600          2021-03-10 15:00:00 CST
1             A      MG01                   10           1615359600          2021-03-10 15:00:00 CST
2             A  Tomcat02                   10           1615359600          2021-03-10 15:00:00 CST
3             A      MG02                    8           1615359660          2021-03-10 15:01:00 CST
4             A  Tomcat01                    8           1615359600          2021-03-10 15:00:00 CST
5             A  Tomcat03                    8           1615359600          2021-03-10 15:00:00 CST
6             A      IG01                    7           1615359660          2021-03-10 15:01:00 CST
7             A      IG02                    5           1615359720          2021-03-10 15:02:00 CST
8             B      MG02                    5           1615360500          2021-03-10 15:15:00 CST
9             B  Tomcat02                    4           1615360500          2021-03-10 15:15:00 CST
10            B      MG01                    3           1615360560          2021-03-10 15:16:00 CST
11            B  Tomcat04                    3           1615360560          2021-03-10 15:16:00 CST
12            B      IG02                    2           1615360500          2021-03-10 15:15:00 CST
13            B  Tomcat01                    1           1615360500          2021-03-10 15:15:00 CST
14            B  Tomcat03                    1           1615360560          2021-03-10 15:16:00 CST)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables if present
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric 'value' and timestamp
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')

# Candidate components
candidates = ['Mysql01','Mysql02','Tomcat01','apache01','apache02','MG01','MG02','IG01','IG02']

# 1) Compute global P5 and P95 per (cmdb_id, log_name) using the full series (no time filtering),
#    but restrict to candidate components to keep results focused (thresholds still use full-time data).
log_full_candidates = log_df[log_df['cmdb_id'].isin(candidates)].copy()

thresholds_log = log_full_candidates.groupby(['cmdb_id','log_name'])['value'].agg(
    P5=lambda x: x.quantile(0.05),
    P95=lambda x: x.quantile(0.95)
).reset_index()

# 2) Define windows in UTC+8 and convert to epoch seconds (timestamps in files are epoch seconds)
windowA_start_local = pd.Timestamp('2021-03-10 15:00:00', tz=tz)
windowA_end_local   = pd.Timestamp('2021-03-10 15:05:00', tz=tz)
windowB_start_local = pd.Timestamp('2021-03-10 15:15:00', tz=tz)
windowB_end_local   = pd.Timestamp('2021-03-10 15:18:00', tz=tz)

windowA_start_epoch = int(windowA_start_local.tz_convert('UTC').timestamp())
windowA_end_epoch   = int(windowA_end_local.tz_convert('UTC').timestamp())
windowB_start_epoch = int(windowB_start_local.tz_convert('UTC').timestamp())
windowB_end_epoch   = int(windowB_end_local.tz_convert('UTC').timestamp())

# Helper to analyze a single window
def analyze_log_window(window_label, start_epoch, end_epoch):
    # Filter rows to candidate components and window
    w = log_full_candidates[
        (log_full_candidates['timestamp'] >= start_epoch) &
        (log_full_candidates['timestamp'] <= end_epoch)
    ].copy()
    if w.empty:
        return pd.DataFrame(columns=[
            'window_label','cmdb_id','log_name','anomaly_type','anomaly_count',
            'earliest_anomaly_ts','extremal_value','threshold_used','severity','full_window_row_count'
        ])
    # counts per series in window
    window_counts = w.groupby(['cmdb_id','log_name']).size().reset_index(name='full_window_row_count')
    # merge thresholds computed from full series
    w = w.merge(thresholds_log, on=['cmdb_id','log_name'], how='left')
    # Drop rows missing thresholds or values
    w = w.dropna(subset=['value','P5','P95'])
    if w.empty:
        return pd.DataFrame(columns=[
            'window_label','cmdb_id','log_name','anomaly_type','anomaly_count',
            'earliest_anomaly_ts','extremal_value','threshold_used','severity','full_window_row_count'
        ])
    # Upper anomalies: value >= P95
    upper = w[w['value'] >= w['P95']].copy()
    if not upper.empty:
        upper_agg = upper.groupby(['cmdb_id','log_name']).agg(
            anomaly_count=('value','size'),
            earliest_anomaly_ts=('timestamp','min'),
            extremal_value=('value','max'),
            threshold_used=('P95','first')
        ).reset_index()
        # severity = extremal_value / threshold_used (guard div by zero)
        upper_agg['severity'] = upper_agg.apply(
            lambda r: (r['extremal_value'] / r['threshold_used']) if r['threshold_used'] not in (0, None, np.nan) else np.inf,
            axis=1
        )
        upper_agg['anomaly_type'] = 'upper'
    else:
        upper_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_anomaly_ts','extremal_value','threshold_used','severity','anomaly_type'])
    # Lower anomalies: value <= P5
    lower = w[w['value'] <= w['P5']].copy()
    if not lower.empty:
        lower_agg = lower.groupby(['cmdb_id','log_name']).agg(
            anomaly_count=('value','size'),
            earliest_anomaly_ts=('timestamp','min'),
            extremal_value=('value','min'),
            threshold_used=('P5','first')
        ).reset_index()
        # severity = threshold_used / extremal_value (guard div by zero)
        lower_agg['severity'] = lower_agg.apply(
            lambda r: (r['threshold_used'] / r['extremal_value']) if r['extremal_value'] not in (0, None, np.nan) else np.inf,
            axis=1
        )
        lower_agg['anomaly_type'] = 'lower'
    else:
        lower_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_anomaly_ts','extremal_value','threshold_used','severity','anomaly_type'])
    combined = pd.concat([upper_agg, lower_agg], ignore_index=True, sort=False)
    if combined.empty:
        return combined
    # attach full_window_row_count
    combined = combined.merge(window_counts, on=['cmdb_id','log_name'], how='left').fillna({'full_window_row_count':0})
    combined['window_label'] = window_label
    return combined[[
        'window_label','cmdb_id','log_name','anomaly_type','anomaly_count',
        'earliest_anomaly_ts','extremal_value','threshold_used','severity','full_window_row_count'
    ]]

# Analyze both windows and cache
log_anomalies_A = analyze_log_window('A', windowA_start_epoch, windowA_end_epoch)
log_anomalies_B = analyze_log_window('B', windowB_start_epoch, windowB_end_epoch)

log_anomalies_both = pd.concat([log_anomalies_A, log_anomalies_B], ignore_index=True, sort=False)
# Convert earliest timestamp to local readable string
if not log_anomalies_both.empty:
    log_anomalies_both['earliest_anomaly_timestamp_local'] = log_anomalies_both['earliest_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # sort by severity desc and limit to top 20 for detailed anomalies output
    log_anomalies_compact = log_anomalies_both.sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)
    # select and rename extremal column to meet requested name
    log_anomalies_compact = log_anomalies_compact[[
        'window_label','cmdb_id','log_name','anomaly_type','anomaly_count',
        'earliest_anomaly_timestamp_local','extremal_value','threshold_used','severity','full_window_row_count'
    ]]
else:
    log_anomalies_compact = pd.DataFrame(columns=[
        'window_label','cmdb_id','log_name','anomaly_type','anomaly_count',
        'earliest_anomaly_timestamp_local','extremal_value','threshold_used','severity','full_window_row_count'
    ])

# 2) Per-component compact summary: window_label, cmdb_id, total_log_anomaly_count, earliest_log_anomaly_timestamp_local
if not log_anomalies_both.empty:
    comp_summary_log = log_anomalies_both.groupby(['window_label','cmdb_id']).agg(
        total_log_anomaly_count=('anomaly_count','sum'),
        earliest_log_anomaly_ts=('earliest_anomaly_ts','min')
    ).reset_index()
    comp_summary_log['earliest_log_anomaly_timestamp_local'] = comp_summary_log['earliest_log_anomaly_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    comp_summary_log = comp_summary_log[[
        'window_label','cmdb_id','total_log_anomaly_count','earliest_log_anomaly_timestamp_local'
    ]].sort_values(['window_label','total_log_anomaly_count'], ascending=[True, False]).head(20).reset_index(drop=True)
else:
    comp_summary_log = pd.DataFrame(columns=['window_label','cmdb_id','total_log_anomaly_count','earliest_log_anomaly_timestamp_local'])

# Explicitly report if error_logs.csv is empty
error_logs_empty = (error_df.shape[0] == 0)
# small descriptor for error logs: either empty or show up to 5 distinct message samples (truncated)
if error_logs_empty:
    error_logs_note = "error_logs.csv is empty (no raw error messages)."
else:
    # capture up to 5 distinct messages truncated
    samples = pd.Series(pd.unique(error_df['message'].dropna())).astype(str)[:5].apply(lambda s: (s[:200] + '...') if len(s)>200 else s).tolist()
    error_logs_note = f"error_logs.csv has {error_df.shape[0]} rows; sample messages: " + " | ".join(samples)

# Final compact outputs (variables to display)
log_anomalies_compact, comp_summary_log, error_logs_empty, error_logs_note

```
Out[5]:
```
Summary of log.csv anomaly scan (candidate components, two UTC+8 windows):

Quick facts
- Windows analyzed:
  - A: 2021-03-10 15:00:00 – 15:05:00 (UTC+8)
  - B: 2021-03-10 15:15:00 – 15:18:00 (UTC+8)
- Global P5/P95 thresholds were computed from the full log series per (cmdb_id, log_name).
- Note: error_logs.csv is empty (no raw error messages) — nothing to inspect there.

Top detailed anomalies (up to 20 shown)
- Many entries involve log.error_count with extremal_value = 0 and threshold = 0, producing infinite severity (division-by-zero). This is a data-quality/threshold artifact; treat these infinite severities cautiously.
- Significant, finite anomalies (examples):
  - Window A, Mysql01, log.row_count — upper anomaly: extremal_value 43 vs P95 26.6, severity ≈ 1.62, anomaly_count=1 (6 rows in window).
  - Window A, apache02, log.row_count — lower anomaly: extremal_value 37 vs P5 48.1, severity = 1.30, anomaly_count=1 (6 rows).
  - Window A, Tomcat01, log.gc_pause_max — lower anomaly: extremal_value 0.03 vs P5 0.0315, severity = 1.05, anomaly_count=1 (5 rows).
  - Window A, Tomcat01, log.row_count — lower anomaly: 353 vs P5 362, severity ≈ 1.03, anomaly_count=1 (6 rows).
- Many log.error_count rows show both "upper" and "lower" flagged due to thresholds of zero; these need inspection to understand if zeros are meaningful or an artifact.

Per-component summary (top components)
- Window A totals (most anomalous): Mysql01 (15 anomalies), Tomcat01 (14), apache02 (13), apache01 (12).
- Window B totals: Tomcat01 (9), Mysql01 (8), apache01 (8), apache02 (8).

Interpretation and recommended next steps
1. Data-quality check: Investigate why log.error_count series have P5/P95 = 0 (leading to infinite severities). Confirm whether zero values are valid (no errors) or reflect missing/instrumentation issues.
2. Prioritize the finite, high-severity log anomalies for operational checks:
   - Mysql01 (high log.row_count spike in Window A): inspect MySQL logs/queries and request throughput at ~15:00.
   - apache02 (log.row_count drop in Window A): check ingress/load-balancer and Apache request counts around ~15:03.
   - Tomcat01 (gc_pause and row_count small deviations): inspect Tomcat logs and GC around 15:02–15:05 and ~15:15.
3. Correlate these log findings with the metric and trace anomalies previously detected (e.g., MySQL disk/I/O and Tomcat CPU/trace row_count drops) to find root cause.
4. If desired, I can extract the raw log time series for any specific (cmdb_id, log_name) (e.g., Mysql01/log.row_count or apache02/log.row_count) for detailed inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(   window_label   cmdb_id          log_name anomaly_type  anomaly_count earliest_anomaly_timestamp_local  extremal_value  threshold_used  severity  full_window_row_count
0             A   Mysql01   log.error_count        upper              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
1             A  Tomcat01   log.error_count        upper              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
2             A  apache02   log.error_count        upper              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
3             A  apache01   log.error_count        upper              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
4             A   Mysql01   log.error_count        lower              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
5             A  Tomcat01   log.error_count        lower              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
6             B  apache01   log.error_count        lower              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
7             B   Mysql01   log.error_count        upper              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
8             A  apache02   log.error_count        lower              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
9             A  apache01   log.error_count        lower              6          2021-03-10 15:00:00 CST            0.00          0.0000       inf                      6
10            B   Mysql01   log.error_count        lower              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
11            B  apache02   log.error_count        upper              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
12            B  apache01   log.error_count        upper              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
13            B  Tomcat01   log.error_count        upper              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
14            B  apache02   log.error_count        lower              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
15            B  Tomcat01   log.error_count        lower              4          2021-03-10 15:15:00 CST            0.00          0.0000       inf                      4
16            A   Mysql01     log.row_count        upper              1          2021-03-10 15:00:00 CST           43.00         26.6000  1.616541                      6
17            A  apache02     log.row_count        lower              1          2021-03-10 15:03:00 CST           37.00         48.1000  1.300000                      6
18            A  Tomcat01  log.gc_pause_max        lower              1          2021-03-10 15:05:00 CST            0.03          0.0315  1.050000                      5
19            A  Tomcat01     log.row_count        lower              1          2021-03-10 15:02:00 CST          353.00        362.0000  1.025496                      6,   window_label   cmdb_id  total_log_anomaly_count earliest_log_anomaly_timestamp_local
0            A   Mysql01                       15              2021-03-10 15:00:00 CST
1            A  Tomcat01                       14              2021-03-10 15:00:00 CST
2            A  apache02                       13              2021-03-10 15:00:00 CST
3            A  apache01                       12              2021-03-10 15:00:00 CST
4            B  Tomcat01                        9              2021-03-10 15:15:00 CST
5            B   Mysql01                        8              2021-03-10 15:15:00 CST
6            B  apache01                        8              2021-03-10 15:15:00 CST
7            B  apache02                        8              2021-03-10 15:15:00 CST, True, 'error_logs.csv is empty (no raw error messages).')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached dataframes and timezone if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric and proper types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# 1) Candidate metric components and substrings for KPI selection
metric_candidates = ['Mysql01','Mysql02','Tomcat01']
substrs = ['Innodb','LOCALDISK','DSK','Row Lock','Row Lock Time','CPU','CPUidle','CPU-0','JVM']

# Prepare candidate metrics (full series for thresholding)
metrics_candidates_full = metric_df[metric_df['cmdb_id'].isin(metric_candidates)].copy()

# 2) Auto-select KPIs whose name contains any of the substrings (case-insensitive)
pattern = '|'.join([s.replace(' ','\\s*') for s in substrs])
sel_mask = metrics_candidates_full['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
selected_kpis = metrics_candidates_full[sel_mask][['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# Compute global thresholds (P5,P95) per (cmdb_id,kpi_name) using full series
thresholds_kpis = metrics_candidates_full.groupby(['cmdb_id','kpi_name'])['value'].agg(
    P5=lambda x: x.quantile(0.05),
    P95=lambda x: x.quantile(0.95)
).reset_index()

# Keep only thresholds for selected KPIs
thresholds_kpis = thresholds_kpis.merge(selected_kpis, on=['cmdb_id','kpi_name'], how='inner')

# 2b) Filter metric rows to the incident window (UTC+8) 2021-03-10 15:00:00 – 15:30:00
win_start_local = pd.Timestamp('2021-03-10 15:00:00', tz=tz)
win_end_local   = pd.Timestamp('2021-03-10 15:30:00', tz=tz)
win_start_epoch = int(win_start_local.tz_convert('UTC').timestamp())
win_end_epoch   = int(win_end_local.tz_convert('UTC').timestamp())

metric_window = metrics_candidates_full[
    (metrics_candidates_full['timestamp'] >= win_start_epoch) &
    (metrics_candidates_full['timestamp'] <= win_end_epoch)
].copy()

# Keep only selected KPIs in the window
metric_window = metric_window.merge(selected_kpis, on=['cmdb_id','kpi_name'], how='inner')

# Attach thresholds to windowed data
metric_window = metric_window.merge(thresholds_kpis, on=['cmdb_id','kpi_name'], how='left')

# Drop rows without thresholds or values
metric_window = metric_window.dropna(subset=['value','P5','P95'])

# 2c) Mark anomalies in window
metric_window['is_upper'] = metric_window['value'] >= metric_window['P95']
metric_window['is_lower'] = metric_window['value'] <= metric_window['P5']

# full_time_window_row_count per component-KPI
full_counts = metric_window.groupby(['cmdb_id','kpi_name']).size().reset_index(name='full_time_window_row_count')

# Helper to group consecutive-minute segments for anomalies (expects minute-granular timestamps)
def group_consecutive_segments(df, ts_col='timestamp'):
    if df.empty:
        return pd.DataFrame(columns=['segment_start_ts','segment_end_ts','segment_length'])
    df = df.sort_values(ts_col).drop_duplicates(subset=[ts_col])
    ts = df[ts_col].astype(int).values
    if len(ts) == 0:
        return pd.DataFrame(columns=['segment_start_ts','segment_end_ts','segment_length'])
    diffs = np.diff(ts, prepend=ts[0])
    new_seg = (diffs != 60)
    seg_id = np.cumsum(new_seg)
    df = df.assign(segment_id=seg_id)
    segs = df.groupby('segment_id').agg(
        segment_start_ts = (ts_col, 'min'),
        segment_end_ts = (ts_col, 'max'),
        segment_length = (ts_col, lambda x: int(len(x)))
    ).reset_index(drop=True)
    return segs

# 2d) For each selected (cmdb_id,kpi_name) and anomaly_type, group consecutive anomalous minutes into segments
segments_list = []

# Iterate over thresholds_kpis which contains P5 and P95
for _, row in thresholds_kpis.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    P5 = float(row['P5'])
    P95 = float(row['P95'])
    sub = metric_window[(metric_window['cmdb_id']==cmdb) & (metric_window['kpi_name']==kpi)].copy()
    if sub.empty:
        continue
    # Upper anomalies
    up = sub[sub['is_upper']].copy()
    if not up.empty:
        up_segs = group_consecutive_segments(up, ts_col='timestamp')
        for _, s in up_segs.iterrows():
            seg_rows = up[(up['timestamp'] >= s['segment_start_ts']) & (up['timestamp'] <= s['segment_end_ts'])]
            extremal = seg_rows['value'].max()
            severity = extremal / P95 if P95 not in (0, None, np.nan) else np.inf
            full_count_row = full_counts[(full_counts['cmdb_id']==cmdb)&(full_counts['kpi_name']==kpi)]
            full_count = int(full_count_row['full_time_window_row_count'].iloc[0]) if not full_count_row.empty else int(seg_rows.shape[0])
            segments_list.append({
                'source':'metric',
                'cmdb_id':cmdb,
                'kpi_or_trace':kpi,
                'anomaly_type':'upper',
                'segment_start_ts':int(s['segment_start_ts']),
                'segment_end_ts':int(s['segment_end_ts']),
                'segment_length_minutes':int(s['segment_length']),
                'extremal_value':float(extremal),
                'threshold_used':float(P95),
                'severity':float(severity),
                'full_time_window_row_count': full_count
            })
    # Lower anomalies
    lo = sub[sub['is_lower']].copy()
    if not lo.empty:
        lo_segs = group_consecutive_segments(lo, ts_col='timestamp')
        for _, s in lo_segs.iterrows():
            seg_rows = lo[(lo['timestamp'] >= s['segment_start_ts']) & (lo['timestamp'] <= s['segment_end_ts'])]
            extremal = seg_rows['value'].min()
            severity = P5 / extremal if extremal not in (0, None, np.nan) else np.inf
            full_count_row = full_counts[(full_counts['cmdb_id']==cmdb)&(full_counts['kpi_name']==kpi)]
            full_count = int(full_count_row['full_time_window_row_count'].iloc[0]) if not full_count_row.empty else int(seg_rows.shape[0])
            segments_list.append({
                'source':'metric',
                'cmdb_id':cmdb,
                'kpi_or_trace':kpi,
                'anomaly_type':'lower',
                'segment_start_ts':int(s['segment_start_ts']),
                'segment_end_ts':int(s['segment_end_ts']),
                'segment_length_minutes':int(s['segment_length']),
                'extremal_value':float(extremal),
                'threshold_used':float(P5),
                'severity':float(severity),
                'full_time_window_row_count': full_count
            })

metric_segments = pd.DataFrame(segments_list)

# Convert metric segment timestamps to local readable strings
if not metric_segments.empty:
    metric_segments['segment_start_ts_local'] = metric_segments['segment_start_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    metric_segments['segment_end_ts_local'] = metric_segments['segment_end_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

# 3) Trace extraction for two specified trace pairs
trace_pairs = [
    ('MG01','trace.to_dockerA2.row_count'),
    ('Tomcat04','trace.self.row_count')
]

# Compute thresholds for these trace pairs using full trace.csv
thresholds_trace_pairs = trace_df[trace_df['cmdb_id'].isin([tp[0] for tp in trace_pairs])].groupby(['cmdb_id','trace_name'])['value'].agg(
    P5=lambda x: x.quantile(0.05),
    P95=lambda x: x.quantile(0.95)
).reset_index()

# Window for traces: 2021-03-10 15:00:00 – 15:05:00 UTC+8
trace_win_start_epoch = int(pd.Timestamp('2021-03-10 15:00:00', tz=tz).tz_convert('UTC').timestamp())
trace_win_end_epoch   = int(pd.Timestamp('2021-03-10 15:05:00', tz=tz).tz_convert('UTC').timestamp())

trace_segments_list = []

for cmdb, tname in trace_pairs:
    thr = thresholds_trace_pairs[(thresholds_trace_pairs['cmdb_id']==cmdb)&(thresholds_trace_pairs['trace_name']==tname)]
    if thr.empty:
        continue
    P5 = float(thr['P5'].iloc[0])
    # filter trace to window and pair
    subt = trace_df[(trace_df['cmdb_id']==cmdb)&(trace_df['trace_name']==tname)&
                    (trace_df['timestamp']>=trace_win_start_epoch)&(trace_df['timestamp']<=trace_win_end_epoch)].copy()
    if subt.empty:
        continue
    full_count = int(subt.shape[0])
    # Mark row_count_drop anomalies where value <= P5
    rc_rows = subt[subt['value'] <= P5].copy()
    if rc_rows.empty:
        continue
    rc_segs = group_consecutive_segments(rc_rows, ts_col='timestamp')
    for _, s in rc_segs.iterrows():
        seg_rows = rc_rows[(rc_rows['timestamp'] >= s['segment_start_ts']) & (rc_rows['timestamp'] <= s['segment_end_ts'])]
        extremal = seg_rows['value'].min()
        severity = P5 / extremal if extremal not in (0, None, np.nan) else np.inf
        trace_segments_list.append({
            'source':'trace',
            'cmdb_id':cmdb,
            'kpi_or_trace':tname,
            'anomaly_type':'row_count_drop',
            'segment_start_ts':int(s['segment_start_ts']),
            'segment_end_ts':int(s['segment_end_ts']),
            'segment_length_minutes':int(s['segment_length']),
            'extremal_value':float(extremal),
            'threshold_used':float(P5),
            'severity':float(severity),
            'full_time_window_row_count': full_count
        })

trace_segments = pd.DataFrame(trace_segments_list)

# Convert trace segment times to local readable strings
if not trace_segments.empty:
    trace_segments['segment_start_ts_local'] = trace_segments['segment_start_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    trace_segments['segment_end_ts_local'] = trace_segments['segment_end_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )

# 4) Combine metric and trace segments into a single fault-segments table for ranking by severity
combined_segments = pd.concat([metric_segments, trace_segments], ignore_index=True, sort=False)

# Keep compact selected columns and sort by severity desc, limit to top 20
if not combined_segments.empty:
    display_cols = [
        'source','cmdb_id','kpi_or_trace','anomaly_type',
        'segment_start_ts_local','segment_end_ts_local','segment_length_minutes',
        'extremal_value','threshold_used','severity','full_time_window_row_count'
    ]
    combined_compact = combined_segments.sort_values(by='severity', ascending=False)[display_cols].head(20).reset_index(drop=True)
else:
    combined_compact = pd.DataFrame(columns=[
        'source','cmdb_id','kpi_or_trace','anomaly_type',
        'segment_start_ts_local','segment_end_ts_local','segment_length_minutes',
        'extremal_value','threshold_used','severity','full_time_window_row_count'
    ])

# Also provide a compact per-component summary for the metric candidates (top 20)
if not metric_segments.empty:
    comp_summary = metric_segments.groupby('cmdb_id').agg(
        total_metric_fault_segments=('kpi_or_trace','nunique'),
        earliest_fault_ts=('segment_start_ts','min')
    ).reset_index()
    comp_summary['earliest_fault_ts_local'] = comp_summary['earliest_fault_ts'].apply(
        lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    comp_summary = comp_summary.sort_values(by='total_metric_fault_segments', ascending=False).head(20)[
        ['cmdb_id','total_metric_fault_segments','earliest_fault_ts_local']
    ].reset_index(drop=True)
else:
    comp_summary = pd.DataFrame(columns=['cmdb_id','total_metric_fault_segments','earliest_fault_ts_local'])

# Final compact outputs (store full intermediates in variables; return compact summaries)
combined_compact, comp_summary

```
Out[6]:
```
Summary of fault-segment extraction (focused on Mysql01, Mysql02, Tomcat01 + two trace pairs):

Key results
- Metric fault-segments (selected KPIs matching Innodb/LOCALDISK/DSK/Row Lock/CPU/JVM):
  - Mysql01: 80 distinct fault segments detected (earliest segment at 2021-03-10 15:00:00 UTC+8).
  - Mysql02: 80 distinct fault segments detected (earliest segment at 2021-03-10 15:00:00 UTC+8).
  - Tomcat01: 38 distinct fault segments detected (earliest segment at 2021-03-10 15:00:00 UTC+8).
- Top returned segments (limited to 20 shown) are dominated by Tomcat01 LOCALDISK/DSK KPIs showing many 1-minute lower-value segments (extremal_value = 0, threshold = 0 → infinite severity). Examples (all UTC+8):
  - Tomcat01 — OSLinux-...-sdb_DSKTps — many single-minute lower segments at 15:05,15:11,15:15,15:17,15:21,15:25,15:27, etc.
  - Tomcat01 — sdb_DSKRead / sdb_DSKRTps — multiple isolated 1-minute lower segments across the window.
- Trace pairs (MG01 → trace.to_dockerA2.row_count and Tomcat04 → trace.self.row_count) were examined for the 15:00–15:05 window and row_count_drop segments were computed. In the combined top-20 severity ranking the metric segments (many zero-valued local-disk KPIs) dominated; trace segments did not appear in the top-20 combined list by severity (they may exist in cached intermediate results).

Data-quality note
- A large fraction of detected metric segments have threshold_used == 0 and extremal_value == 0, producing infinite severity ratios. This strongly suggests either:
  - genuine zero measurements for those KPIs in the window (e.g., no reads/writes during that minute), or
  - sparse/zero-heavy historical series that make P5/P95 = 0 (instrumentation or normalization issue).
- Because infinite severities distort ranking, treat those entries as data-quality artifacts until the raw series are inspected.

Interpretation — what this confirms
- DB-related signals: Both Mysql01 and Mysql02 show many KPI fault segments across Innodb/DSK/Row-Lock related KPIs with earliest anomalies at the window start (15:00 UTC+8). This aligns with earlier anomaly scans that flagged MySQL disk/I/O and InnoDB lock/time issues.
- Tomcat-related signals: Tomcat01 shows many local-disk DSK lower anomalies (zero values) across the window; earlier scans also flagged Tomcat CPU and trace row_count effects—Tomcat disk anomalies warrant inspection.
- Trace traffic-drop (around ~15:02): trace analysis previously identified row_count drops (MG01 → dockerA2 and Tomcat04 → self) near 15:02. In this fault-segments ranking those trace segments did not out-rank the zero-valued metric segments, but they remain captured in cached intermediates for follow-up.

Recommended next actions (prioritized)
1. Inspect raw time series for top DB KPIs (Mysql01 & Mysql02): Innodb metrics, DSK read/write/rtps, Row Lock / Row Lock Time around 15:00–15:30 to confirm magnitude and continuity of anomalies (not just single-minute zeros).
2. Inspect Tomcat01 raw disk KPIs (sdb DSKRead/DSKTps/DSKRTps) and CPU KPIs at ~15:16 to confirm whether zeros are valid or instrumentation gaps.
3. Inspect the trace raw series for:
   - MG01 → trace.to_dockerA2.row_count (around 15:02)
   - Tomcat04 → trace.self.row_count (around 15:02)
   to confirm the row_count drop shape/duration.
4. Correlate confirmed KPI/trace fault segments with logs (e.g., Mysql slowlogs, Tomcat logs) and infrastructure metrics (disk I/O, network) at the same timestamps to identify root cause (I/O contention, brief traffic drop, or instrumentation issue).
5. If desired, I can now extract the raw minute-series for any selected (cmdb_id,kpi_name) or the two trace pairs for detailed inspection (start/end times and values).

The original code execution output of IPython Kernel is also provided below for reference:

(    source   cmdb_id                                     kpi_or_trace anomaly_type   segment_start_ts_local     segment_end_ts_local  segment_length_minutes  extremal_value  threshold_used  severity  full_time_window_row_count
0   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:25:00 CST  2021-03-10 15:25:00 CST                       1             0.0             0.0       inf                           9
1   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:27:00 CST  2021-03-10 15:27:00 CST                       1             0.0             0.0       inf                           9
2   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:05:00 CST  2021-03-10 15:05:00 CST                       1             0.0             0.0       inf                           9
3   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:11:00 CST  2021-03-10 15:11:00 CST                       1             0.0             0.0       inf                           9
4   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:15:00 CST  2021-03-10 15:15:00 CST                       1             0.0             0.0       inf                           9
5   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:17:00 CST  2021-03-10 15:17:00 CST                       1             0.0             0.0       inf                           9
6   metric  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps        lower  2021-03-10 15:21:00 CST  2021-03-10 15:21:00 CST                       1             0.0             0.0       inf                           9
7   metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:04:00 CST  2021-03-10 15:04:00 CST                       1             0.0             0.0       inf                           9
8   metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:06:00 CST  2021-03-10 15:06:00 CST                       1             0.0             0.0       inf                           9
9   metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:10:00 CST  2021-03-10 15:10:00 CST                       1             0.0             0.0       inf                           9
10  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:16:00 CST  2021-03-10 15:16:00 CST                       1             0.0             0.0       inf                           9
11  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:22:00 CST  2021-03-10 15:22:00 CST                       1             0.0             0.0       inf                           9
12  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:24:00 CST  2021-03-10 15:24:00 CST                       1             0.0             0.0       inf                           9
13  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead        lower  2021-03-10 15:30:00 CST  2021-03-10 15:30:00 CST                       1             0.0             0.0       inf                           9
14  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        lower  2021-03-10 15:04:00 CST  2021-03-10 15:04:00 CST                       1             0.0             0.0       inf                           9
15  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        lower  2021-03-10 15:06:00 CST  2021-03-10 15:06:00 CST                       1             0.0             0.0       inf                           9
16  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        lower  2021-03-10 15:10:00 CST  2021-03-10 15:10:00 CST                       1             0.0             0.0       inf                           9
17  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        lower  2021-03-10 15:16:00 CST  2021-03-10 15:16:00 CST                       1             0.0             0.0       inf                           9
18  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        lower  2021-03-10 15:22:00 CST  2021-03-10 15:22:00 CST                       1             0.0             0.0       inf                           9
19  metric  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps        lower  2021-03-10 15:24:00 CST  2021-03-10 15:24:00 CST                       1             0.0             0.0       inf                           9,     cmdb_id  total_metric_fault_segments  earliest_fault_ts_local
0   Mysql01                           80  2021-03-10 15:00:00 CST
1   Mysql02                           80  2021-03-10 15:00:00 CST
2  Tomcat01                           38  2021-03-10 15:00:00 CST)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes and tz from kernel state
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# Define windows (UTC+8) and convert to epoch seconds (UTC)
metric_win_start_epoch = int(pd.Timestamp('2021-03-10 15:00:00', tz=tz).tz_convert('UTC').timestamp())
metric_win_end_epoch   = int(pd.Timestamp('2021-03-10 15:30:00', tz=tz).tz_convert('UTC').timestamp())
trace_win_start_epoch  = int(pd.Timestamp('2021-03-10 15:00:00', tz=tz).tz_convert('UTC').timestamp())
trace_win_end_epoch    = int(pd.Timestamp('2021-03-10 15:05:00', tz=tz).tz_convert('UTC').timestamp())

# Requested metric series list (cmdb_id, kpi_name)
metric_requests = [
    ('Mysql01', 'Mysql-MySQL_3306_Innodb Row Lock Time'),
    ('Mysql01', 'Mysql-MySQL_3306_Innodb Row Lock Waits'),
    ('Mysql01', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps'),
    ('Mysql01', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps'),
    ('Mysql02', 'Mysql-MySQL_3306_Innodb Row Lock Time'),
    ('Mysql02', 'Mysql-MySQL_3306_Innodb Row Lock Waits'),
    ('Mysql02', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps'),
    ('Mysql02', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps'),
    ('Tomcat01', 'OSLinux-CPU_CPU-0_SingleCpuidle'),
    ('Tomcat01', 'OSLinux-CPU_CPU_CPUWio')
]

# Requested trace series list (cmdb_id, trace_name)
trace_requests = [
    ('MG01', 'trace.to_dockerA2.row_count'),
    ('Tomcat04', 'trace.self.row_count')
]

# Prepare containers
rows_list = []      # will hold per-series rows (limited)
summaries = []      # per-series summary records
thresholds_metrics_req = []  # store global thresholds per requested metric
thresholds_traces_req = []   # store global thresholds per requested trace

# Helper to convert epoch->local string
def epoch_to_local_str(epoch):
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Process metric series
for cmdb, kpi in metric_requests:
    # Compute global thresholds on full metric_df for this series (rule 9)
    full_series = metric_df[(metric_df['cmdb_id']==cmdb) & (metric_df['kpi_name']==kpi)]['value'].dropna()
    if not full_series.empty:
        P5 = float(full_series.quantile(0.05))
        P95 = float(full_series.quantile(0.95))
    else:
        P5 = np.nan; P95 = np.nan
    thresholds_metrics_req.append({'cmdb_id':cmdb,'kpi_name':kpi,'P5':P5,'P95':P95})
    # Filter to metric window and requested series
    sub = metric_df[
        (metric_df['cmdb_id']==cmdb) &
        (metric_df['kpi_name']==kpi) &
        (metric_df['timestamp']>=metric_win_start_epoch) &
        (metric_df['timestamp']<=metric_win_end_epoch)
    ].copy().sort_values('timestamp')
    # Limit rows to 50
    sub_limited = sub.head(50).copy()
    # Add timestamp_local column
    if not sub_limited.empty:
        sub_limited['timestamp_local'] = sub_limited['timestamp'].apply(lambda t: epoch_to_local_str(int(t)))
        # Add series_id
        sub_limited['series_id'] = cmdb + ' | ' + kpi
        # Select desired columns and append to rows_list
        rows_list.append(sub_limited[['timestamp_local','cmdb_id','kpi_name','value','series_id']])
        # summary
        num = int(sub.shape[0])
        min_val = float(sub['value'].min()) if not sub['value'].dropna().empty else np.nan
        max_val = float(sub['value'].max()) if not sub['value'].dropna().empty else np.nan
        earliest = int(sub['timestamp'].min()) if not sub['timestamp'].dropna().empty else None
        earliest_local = epoch_to_local_str(earliest) if earliest is not None else None
    else:
        # empty series in window
        rows_list.append(pd.DataFrame(columns=['timestamp_local','cmdb_id','kpi_name','value','series_id']))
        num = 0; min_val = np.nan; max_val = np.nan; earliest_local = None
    summaries.append({
        'series_id': cmdb + ' | ' + kpi,
        'points_returned': num if num<=50 else 50,
        'min_value': min_val,
        'max_value': max_val,
        'earliest_timestamp_local': earliest_local
    })

# Process trace series
for cmdb, tname in trace_requests:
    # Compute global thresholds on full trace_df for this pair
    full_series = trace_df[(trace_df['cmdb_id']==cmdb) & (trace_df['trace_name']==tname)]['value'].dropna()
    if not full_series.empty:
        P5 = float(full_series.quantile(0.05))
        P95 = float(full_series.quantile(0.95))
    else:
        P5 = np.nan; P95 = np.nan
    thresholds_traces_req.append({'cmdb_id':cmdb,'trace_name':tname,'P5':P5,'P95':P95})
    # Filter to trace window
    sub = trace_df[
        (trace_df['cmdb_id']==cmdb) &
        (trace_df['trace_name']==tname) &
        (trace_df['timestamp']>=trace_win_start_epoch) &
        (trace_df['timestamp']<=trace_win_end_epoch)
    ].copy().sort_values('timestamp')
    sub_limited = sub.head(50).copy()
    if not sub_limited.empty:
        sub_limited['timestamp_local'] = sub_limited['timestamp'].apply(lambda t: epoch_to_local_str(int(t)))
        sub_limited['series_id'] = cmdb + ' | ' + tname
        rows_list.append(sub_limited[['timestamp_local','cmdb_id','trace_name','value','series_id']].rename(columns={'trace_name':'kpi_name'}))
        num = int(sub.shape[0])
        min_val = float(sub['value'].min()) if not sub['value'].dropna().empty else np.nan
        max_val = float(sub['value'].max()) if not sub['value'].dropna().empty else np.nan
        earliest = int(sub['timestamp'].min()) if not sub['timestamp'].dropna().empty else None
        earliest_local = epoch_to_local_str(earliest) if earliest is not None else None
    else:
        rows_list.append(pd.DataFrame(columns=['timestamp_local','cmdb_id','kpi_name','value','series_id']))
        num = 0; min_val = np.nan; max_val = np.nan; earliest_local = None
    summaries.append({
        'series_id': cmdb + ' | ' + tname,
        'points_returned': num if num<=50 else 50,
        'min_value': min_val,
        'max_value': max_val,
        'earliest_timestamp_local': earliest_local
    })

# Concatenate extracted rows into one DataFrame for compact display
extracted_rows = pd.concat(rows_list, ignore_index=True, sort=False)
# Normalize column order
if 'kpi_name' not in extracted_rows.columns:
    extracted_rows = extracted_rows.rename(columns={'kpi_name':'kpi_name'})

# Build summaries DataFrame
series_summaries = pd.DataFrame(summaries)

# Keep outputs compact: for extracted rows, show up to 50 rows per series already limited; but overall may have many rows.
# To keep display bounded, show .head(200) (should be small since limited per series).
extracted_rows_limited = extracted_rows.copy()  # already bounded per series
series_summaries_limited = series_summaries.copy()

# Expose computed global thresholds for requested series (cached) as DataFrames
thresholds_metrics_requested = pd.DataFrame(thresholds_metrics_req)
thresholds_traces_requested = pd.DataFrame(thresholds_traces_req)

# Return compact outputs (variables)
extracted_rows_limited, series_summaries_limited, thresholds_metrics_requested, thresholds_traces_requested

```
Out[7]:
```
Compact extraction summary (UTC+8) — per-series one-line summaries and key observations.

Per-series one-line summaries (series_id — points_returned — min — max — earliest_timestamp_local):
- Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time — 21 pts — min 0.0000 — max 88.4407 — earliest 2021-03-10 15:00:00 CST
- Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Waits — 21 pts — min 0.0000 — max 0.2034 — earliest 2021-03-10 15:00:00 CST
- Mysql01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps — 15 pts — min 0.0000 — max 8.0000 — earliest 2021-03-10 15:01:00 CST
- Mysql01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps — 15 pts — min 3.0000 — max 34.0000 — earliest 2021-03-10 15:01:00 CST
- Mysql02 | Mysql-MySQL_3306_Innodb Row Lock Time — 21 pts — min 0.0167 — max 68.1017 — earliest 2021-03-10 15:00:00 CST
- Mysql02 | Mysql-MySQL_3306_Innodb Row Lock Waits — 21 pts — min 0.0167 — max 1.4500 — earliest 2021-03-10 15:00:00 CST
- Mysql02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps — 15 pts — min 0.0000 — max 5.0000 — earliest 2021-03-10 15:00:00 CST
- Mysql02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps — 15 pts — min 20.0000 — max 472.0000 — earliest 2021-03-10 15:00:00 CST
- Tomcat01 | OSLinux-CPU_CPU-0_SingleCpuidle — 9 pts — min 1.0965 — max 97.2901 — earliest 2021-03-10 15:04:00 CST
- Tomcat01 | OSLinux-CPU_CPU_CPUWio — 16 pts — min 0.0083 — max 1.7510 — earliest 2021-03-10 15:01:00 CST
- MG01 | trace.to_dockerA2.row_count — 6 pts — min 82.0000 — max 191.0000 — earliest 2021-03-10 15:00:00 CST
- Tomcat04 | trace.self.row_count — 6 pts — min 2118.0000 — max 3694.0000 — earliest 2021-03-10 15:00:00 CST

Global thresholds (P5 / P95) for requested series (key values):
- Mysql01 Innodb Row Lock Time: P5=0.0000, P95=54.8167
- Mysql01 Innodb Row Lock Waits: P5=0.0000, P95=0.1000
- Mysql01 sda_DSKBps: P5=0.0000, P95=3.1000
- Mysql01 sdc_DSKBps: P5=3.0000, P95=22.1000
- Mysql02 Innodb Row Lock Time: P5=0.1525, P95=19.4333
- Mysql02 Innodb Row Lock Waits: P5=0.0678, P95=0.7500
- Mysql02 sda_DSKBps: P5=0.0000, P95=1.5000
- Mysql02 sdc_DSKBps: P5=137.6000, P95=384.5000
- Tomcat01 CPU-0 SingleCpuidle: P5=38.66654, P95=97.16234
- Tomcat01 CPUWio: P5=0.017675, P95=0.967925
- MG01 trace.to_dockerA2.row_count: P5=131.70, P95=219.3
- Tomcat04 trace.self.row_count: P5=2589.65, P95=3999.9

Key observations and confirmations
- Trace drops around ~15:02 (Window A) are confirmed:
  - MG01 → trace.to_dockerA2.row_count dropped to 82 at one minute (min=82 < P5=131.70) — confirms a traffic drop.
  - Tomcat04 → trace.self.row_count dropped to 2118 at one minute (min=2118 < P5=2589.65) — confirms the earlier detected trace row_count drop around 15:02.
- MySQL indicators:
  - Innodb Row Lock Time (Mysql01 and Mysql02) show non-zero spikes (max up to ~88 for Mysql01 and ~68 for Mysql02) with multiple data points across the 15:00–15:30 window — consistent with InnoDB contention risk.
  - Mysql02 sdc_DSKBps shows very large max (472) and high P95 (384.5) — indicates heavy disk activity on sdc for Mysql02 in the window.
  - Several MySQL series include zeros (P5=0 for some Mysql01 series), so low/zero values occur and should be interpreted carefully.
- Tomcat01 CPU:
  - CPU idle dropped as low as ~1.10 at one minute (min 1.0965), indicating a minute of very high CPU usage; CPU-wio is low but shows spikes up to ~1.75.

Concise recommendations for confirmation
1. For traces: focus on ~15:02 exact minutes for MG01→dockerA2 and Tomcat04 self — the extracted minute rows show the low values at that timestamp; inspect adjacent minutes (we extracted them) to confirm whether the drop was a single-minute spike or multi-minute event.
2. For MySQL: inspect Innodb Row Lock Time/Waits and sdc DSK metrics across 15:00–15:30 — the extracted series contain multiple non-zero spikes and a high sdc DSKBps on Mysql02; these are strong candidates for DB contention / heavy I/O.
3. For Tomcat01: inspect the minute with CPU idle ~1.10 (very low) to identify CPU-bound activity at that time (15:04 shown in extracted rows; prior anomaly detection also flagged ~15:16 — check both minutes in the raw series).
4. Because some series have zeros and P5=0, verify instrumentation validity for those KPIs before treating infinite severity ratios as true outages.

If you want, I can now present the extracted minute rows for any single series (up to 50 rows) so you can inspect exact timestamps and values (e.g., show the full 21 rows for Mysql01 Innodb Row Lock Time or the 6 trace rows around 15:02).

The original code execution output of IPython Kernel is also provided below for reference:

(             timestamp_local   cmdb_id                               kpi_name      value                                        series_id
0    2021-03-10 15:00:00 CST   Mysql01  Mysql-MySQL_3306_Innodb Row Lock Time    10.0333  Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time
1    2021-03-10 15:01:00 CST   Mysql01  Mysql-MySQL_3306_Innodb Row Lock Time     9.6393  Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time
2    2021-03-10 15:03:00 CST   Mysql01  Mysql-MySQL_3306_Innodb Row Lock Time     1.0333  Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time
3    2021-03-10 15:04:00 CST   Mysql01  Mysql-MySQL_3306_Innodb Row Lock Time     1.0833  Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time
4    2021-03-10 15:06:00 CST   Mysql01  Mysql-MySQL_3306_Innodb Row Lock Time     5.9833  Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time
..                       ...       ...                                    ...        ...                                              ...
176  2021-03-10 15:01:00 CST  Tomcat04                   trace.self.row_count  3304.0000                  Tomcat04 | trace.self.row_count
177  2021-03-10 15:02:00 CST  Tomcat04                   trace.self.row_count  2118.0000                  Tomcat04 | trace.self.row_count
178  2021-03-10 15:03:00 CST  Tomcat04                   trace.self.row_count  2621.0000                  Tomcat04 | trace.self.row_count
179  2021-03-10 15:04:00 CST  Tomcat04                   trace.self.row_count  3694.0000                  Tomcat04 | trace.self.row_count
180  2021-03-10 15:05:00 CST  Tomcat04                   trace.self.row_count  3253.0000                  Tomcat04 | trace.self.row_count

[181 rows x 5 columns],                                             series_id  points_returned  min_value  max_value earliest_timestamp_local
0     Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Time               21     0.0000    88.4407  2021-03-10 15:00:00 CST
1    Mysql01 | Mysql-MySQL_3306_Innodb Row Lock Waits               21     0.0000     0.2034  2021-03-10 15:00:00 CST
2   Mysql01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-...               15     0.0000     8.0000  2021-03-10 15:01:00 CST
3   Mysql01 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-...               15     3.0000    34.0000  2021-03-10 15:01:00 CST
4     Mysql02 | Mysql-MySQL_3306_Innodb Row Lock Time               21     0.0167    68.1017  2021-03-10 15:00:00 CST
5    Mysql02 | Mysql-MySQL_3306_Innodb Row Lock Waits               21     0.0167     1.4500  2021-03-10 15:00:00 CST
6   Mysql02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-...               15     0.0000     5.0000  2021-03-10 15:00:00 CST
7   Mysql02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-...               15    20.0000   472.0000  2021-03-10 15:00:00 CST
8          Tomcat01 | OSLinux-CPU_CPU-0_SingleCpuidle                9     1.0965    97.2901  2021-03-10 15:04:00 CST
9                   Tomcat01 | OSLinux-CPU_CPU_CPUWio               16     0.0083     1.7510  2021-03-10 15:01:00 CST
10                 MG01 | trace.to_dockerA2.row_count                6    82.0000   191.0000  2021-03-10 15:00:00 CST
11                    Tomcat04 | trace.self.row_count                6  2118.0000  3694.0000  2021-03-10 15:00:00 CST,     cmdb_id                                        kpi_name          P5         P95
0   Mysql01           Mysql-MySQL_3306_Innodb Row Lock Time    0.000000   54.816700
1   Mysql01          Mysql-MySQL_3306_Innodb Row Lock Waits    0.000000    0.100000
2   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.000000    3.100000
3   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps    3.000000   22.100000
4   Mysql02           Mysql-MySQL_3306_Innodb Row Lock Time    0.152500   19.433300
5   Mysql02          Mysql-MySQL_3306_Innodb Row Lock Waits    0.067800    0.750000
6   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.000000    1.500000
7   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps  137.600000  384.500000
8  Tomcat01                 OSLinux-CPU_CPU-0_SingleCpuidle   38.666540   97.162340
9  Tomcat01                          OSLinux-CPU_CPU_CPUWio    0.017675    0.967925,     cmdb_id                   trace_name       P5     P95
0      MG01  trace.to_dockerA2.row_count   131.70   219.3
1  Tomcat04         trace.self.row_count  2589.65  3999.9)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```